In [2]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import re
from pandas.api.types import (
  is_numeric_dtype,
  is_string_dtype,
  is_datetime64_any_dtype
)

def run_data_cleaner():

  audit_log = []
  rows_removed = False
  rows_flagged = False

  # Detailed report tracking — NEW
  report = {
      "file_info"          : {},
      "shape"              : {},
      "column_std"         : {},
      "id_protection"      : [],
      "duplicates"         : {},
      "datatype_changes"   : [],
      "missing_handling"   : [],
      "text_normalization" : [],
      "skipped_steps"      : [],
      "final"              : {},
  }

  def log_action(msg):
      audit_log.append(f"[APPROVED] {msg}")

  def integrity_snapshot(stage_name, df, expected_rows):

      if df.shape[0] != expected_rows and not rows_removed:
          raise ValueError(f"Row integrity violated at stage: {stage_name}")
      print(f"✔ Integrity check passed after {stage_name}")

  print("\nUpload your dataset (CSV, Excel, JSON, Parquet, TXT supported)...")
  from google.colab import files
  uploaded = files.upload()
  file_name = list(uploaded.keys())[0]

  print(f"\nDetected file: {file_name}")

  if file_name.lower().endswith(".csv"):
      for enc in ["utf-8", "latin-1", "cp1252"]:
          try:
              df = pd.read_csv(file_name, encoding=enc, low_memory=False)
              print(f"Loaded CSV using encoding: {enc}")
              report["file_info"]["encoding"] = enc
              break
          except:
              continue
  elif file_name.lower().endswith((".xlsx", ".xls")):
      df = pd.read_excel(file_name)
  elif file_name.lower().endswith(".json"):
      df = pd.read_json(file_name)
  elif file_name.lower().endswith(".parquet"):
      df = pd.read_parquet(file_name)
  elif file_name.lower().endswith(".txt"):
      df = pd.read_csv(file_name, sep=None, engine='python')
  else:
      raise ValueError("Unsupported file type.")

  original_row_count = df.shape[0]
  original_column_set = set(df.columns)
  original_columns_list = list(df.columns)

  report["file_info"]["filename"]     = file_name
  report["file_info"]["loaded_at"]    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  report["shape"]["original_rows"]    = original_row_count
  report["shape"]["original_cols"]    = df.shape[1]
  report["shape"]["original_columns"] = list(df.columns)

  print("\n=========== DATA SUMMARY ===========")
  print("Rows:", df.shape[0])
  print("Columns:", df.shape[1])
  print("Memory (MB):", round(df.memory_usage(deep=True).sum()/1024**2,2))
  print("\nData Types:\n", df.dtypes.value_counts())
  print("\nMissing Values per Column:\n", df.isnull().sum())
  print("\nExact Duplicate Rows:", df.duplicated().sum())
  print("=====================================")

  integrity_snapshot("Initial Load", df, original_row_count)

  print("\n=========== COLUMN-WISE DATA PROFILE ===========")

  profile_df = pd.DataFrame({
      "Column Name"  : df.columns,
      "Data Type"    : df.dtypes.astype(str).values,
      "Missing Values": df.isnull().sum().values,
      "Missing %"    : round((df.isnull().sum() / len(df)) * 100, 2).values,
      "Unique Values": df.nunique().values
  })

  print(profile_df)
  print("===============================================")

  print("\n=========== MISSING ROW CHECK ===========")

  total_rows_before = df.shape[0]

  empty_rows  = df[df.isnull().all(axis=1)]
  empty_count = len(empty_rows)

  print(f"Fully empty rows: {empty_count}")

  if empty_count > 0:
      if input("Remove fully empty rows? (y/n): ").lower() == "y":
          df = df.dropna(how='all')
          rows_removed = True   # FIX 1
          print(f"Removed {empty_count} fully empty rows.")
          log_action(f"Removed {empty_count} fully empty rows.")
          report["shape"]["empty_rows_removed"] = empty_count

  exact_duplicate_count = df.duplicated().sum()

  print(f"\nExact duplicate rows (fully identical): {exact_duplicate_count}")

  if exact_duplicate_count > 0:
      print("Only fully identical rows will be removed.")
      print("If 3 identical rows exist → 1 will be kept, others removed.")
      print("Rows with even 1 different value will NOT be removed.")

      if input("Remove exact duplicate rows? (y/n): ").lower() == "y":
          df = df.drop_duplicates(keep='first')
          rows_removed = True   # FIX 1
          removed_count = total_rows_before - df.shape[0]
          print(f"Removed {removed_count} duplicate rows (kept one copy).")
          log_action(f"Removed {removed_count} exact duplicate rows (kept one copy).")
      else:
          log_action("Exact duplicate removal skipped.")
  else:
      print("No exact duplicate rows detected.")

  print("\nNOTE:")
  print("Rows with partial missing values are NOT removed here.")
  print("They will be handled in the missing value stage.")

  integrity_snapshot("Missing Row Check", df, original_row_count)

  print("\n=========== COLUMN STANDARDIZATION ===========")

  print("""
Column Standardization ensures consistency and clean structure.
• Remove leading/trailing spaces from column names
• Convert all column names to lowercase
• Replace spaces with underscores (snake_case format)
• Improve compatibility with Python, SQL, and ML pipelines
• Prevent errors caused by special characters or inconsistent casing
""")

  print("Current Columns:")
  print(list(df.columns))

  col_std_applied = False
  if input("\nStandardize column names? (y/n): ").lower() == 'y':
      old_cols = list(df.columns)
      df.columns = (
          df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_")
      )
      new_cols = list(df.columns)
      original_column_set = set(df.columns)
      col_std_applied = True
      log_action("Column names standardized (lowercase + snake_case + trimmed).")
      report["column_std"]["applied"]  = True
      report["column_std"]["before"]   = old_cols
      report["column_std"]["after"]    = new_cols
      report["column_std"]["renamed"]  = [(o, n) for o, n in zip(old_cols, new_cols) if o != n]

      print("\nUpdated Columns:")
      print(list(df.columns))
  else:
      log_action("Column standardization skipped.")
      report["column_std"]["applied"] = False
      report["skipped_steps"].append("Column standardization — skipped by user")

  original_dtypes       = df.dtypes.astype(str).to_dict()
  original_columns_list = list(df.columns)

  integrity_snapshot("Column Standardization", df, original_row_count)

  id_columns = []

  print("\n=========== ID COLUMN DETECTION ===========")

  print("""
  This step detects potential Unique Identifier (Primary Key) columns.
  Detection logic is DATA-DRIVEN and NAME-AWARE:
  • Column name keyword match (weak signal)
  • High uniqueness ratio (strong signal)
  • Null ratio check
  • Date-pattern protection (prevents invoicedate false positives)
  Only columns with sufficient ID score are suggested.
  """)

  identifier_keywords = [
      "id", "code", "key", "number",
      "invoice", "ref", "uuid"
  ]

  for col in df.columns:

      non_null = df[col].dropna()
      if len(non_null) == 0:
          continue

      uniqueness_ratio = non_null.nunique() / len(non_null)
      null_ratio       = df[col].isnull().mean()
      col_lower        = col.lower()

      keyword_match = any(
          re.search(rf'(^|_){kw}($|_)', col_lower)
          for kw in identifier_keywords
      )

      datetime_ratio = pd.to_datetime(
          df[col], errors="coerce"
      ).notna().mean()

      looks_like_date = datetime_ratio > 0.90

      id_score = 0
      reasons  = []

      if keyword_match:
          id_score += 1
          reasons.append("keyword match")

      if uniqueness_ratio > 0.98:
          id_score += 2
          reasons.append("high uniqueness (>98%)")

      if null_ratio == 0:
          id_score += 1
          reasons.append("no missing values")

      if looks_like_date:
          id_score -= 2
          reasons.append("looks like datetime column")

      if id_score >= 2:

          print(f"\nCandidate Column: {col}")
          print("------------------------------------------------")
          print(f"Non-null values : {len(non_null)}")
          print(f"Unique values   : {non_null.nunique()}")
          print(f"Uniqueness Ratio: {round(uniqueness_ratio,4)}")
          print(f"Null Ratio      : {round(null_ratio,4)}")
          print(f"Datetime Ratio  : {round(datetime_ratio,4)}")
          print(f"ID Score        : {id_score}")
          print(f"Reasons         : {', '.join(reasons)}")

          print("""
If protected:
• Column will be locked as STRING
• Skipped in datatype conversion
• Excluded from text normalization
• Treated as stable business key
""")

          if input(f"Protect {col} as ID? (y/n): ").lower() == "y":

              df[col] = df[col].astype("string")
              id_columns.append(col)

              log_action(
                  f"{col} protected as ID "
                  f"(score={id_score}, uniqueness={round(uniqueness_ratio,4)})."
              )

              report["id_protection"].append({
                  "column": col,
                  "protected": True,
                  "id_score": id_score,
                  "uniqueness_ratio": round(uniqueness_ratio, 4),
                  "null_ratio": round(null_ratio, 4),
                  "datetime_ratio": round(datetime_ratio, 4),
                  "reason": ", ".join(reasons)
              })

          else:

              log_action(f"{col} not protected as ID (user declined).")

              report["id_protection"].append({
                  "column": col,
                  "protected": False,
                  "id_score": id_score,
                  "reason": "user declined"
              })

  print("\n=============================================")
  integrity_snapshot("ID Protection", df, original_row_count)

  print("\n=========== DUPLICATE HANDLING ===========")

  total_rows_before  = df.shape[0]
  duplicate_count    = df.duplicated().sum()

  print(f"Total rows before duplicate handling: {total_rows_before}")
  print(f"Exact duplicate rows detected: {duplicate_count}")

  if duplicate_count > 0:

      print("\nOptions:")
      print("1 - Remove duplicates (keep first occurrence)")
      print("2 - Flag duplicates (add 'is_duplicate' column)")
      print("3 - Skip duplicate handling")

      choice = input("Choose (1/2/3): ")

      if choice == "1":
          confirm = input("Are you sure you want to remove duplicate rows? (y/n): ")

          if confirm.lower() == "y":
              df = df.drop_duplicates(keep='first')
              rows_removed = True
              total_rows_after = df.shape[0]

              print(f"\nRows after duplicate removal: {total_rows_after}")
              print(f"Rows removed: {total_rows_before - total_rows_after}")

              log_action(
                  f"Duplicate rows removed."
                  f"Before={total_rows_before}, After={total_rows_after}"
              )
              report["duplicates"] = {
                  "action": "removed",
                  "count_found": int(duplicate_count),
                  "rows_before": total_rows_before,
                  "rows_after": total_rows_after,
                  "rows_removed": total_rows_before - total_rows_after,
              }
          else:
              print("Duplicate removal cancelled.")
              total_rows_after = total_rows_before
              log_action("Duplicate removal cancelled by user.")
              report["duplicates"] = {"action": "cancelled by user", "count_found": int(duplicate_count)}

      elif choice == "2":
          df["is_duplicate"] = df.duplicated(keep=False)
          rows_flagged  = True
          total_rows_after = df.shape[0]

          print("\nDuplicates flagged. No rows removed.")
          print(f"Total rows remain: {total_rows_after}")

          log_action("Duplicate rows flagged (no deletion).")
          report["duplicates"] = {
              "action": "flagged only",
              "count_found" : int(duplicate_count),
              "flag_column" : "is_duplicate",
              "rows_removed": 0,
          }

      else:
          total_rows_after = total_rows_before
          print("\nDuplicate handling skipped.")
          log_action("Duplicate handling skipped.")
          report["duplicates"] = {"action": "skipped by user", "count_found": int(duplicate_count)}

  else:
      print("No duplicate rows found.")
      total_rows_after = total_rows_before
      report["duplicates"] = {"action": "none found", "count_found": 0}

  integrity_snapshot("Duplicate Handling", df, original_row_count)


  print("\n=========== DATATYPE REVIEW (DATA-BASED) ===========")

  print("""
This step analyzes actual column values (not column names)
to determine the most appropriate datatype.

No automatic conversions will occur.
You will be prompted before any change.
ALL columns are checked — including numeric and datetime ones.
""")

  for col in df.columns:

      if col in id_columns:
          print(f"\n{col} — PROTECTED ID column | Current type: {df[col].dtype} | No conversion")
          report["datatype_changes"].append({
              "column": col,
              "current_type" : str(df[col].dtype),
              "suggested": "string (ID protected)",
              "action": "skipped — ID column",
              "reason": "column is protected as identifier"
          })
          continue

      series       = df[col]
      current_dtype = str(df[col].dtype)

      cleaned = (
          series.astype(str)
          .str.strip()
          .str.replace(",", "", regex=False)
          .str.replace("₹", "", regex=False)
          .str.replace("$", "", regex=False)
      )

      numeric_test  = pd.to_numeric(cleaned, errors="coerce")
      numeric_ratio = numeric_test.notna().mean()

      datetime_test  = pd.to_datetime(cleaned, errors="coerce")
      datetime_ratio = datetime_test.notna().mean()

      time_test  = pd.to_datetime(cleaned, format="%H:%M:%S", errors="coerce")
      time_ratio = time_test.notna().mean()

      if numeric_ratio >= 0.80:
          suggested_type = "numeric"
      elif datetime_ratio >= 0.80:
          suggested_type = "datetime"
      elif time_ratio >= 0.80:
          suggested_type = "time"
      else:
          suggested_type = "string"

      mismatch = False

      if suggested_type == "numeric" and not is_numeric_dtype(df[col]):
          mismatch = True
      elif suggested_type == "datetime" and not is_datetime64_any_dtype(df[col]):
          mismatch = True
      elif suggested_type == "string" and is_numeric_dtype(df[col]):
          mismatch = True
      elif suggested_type == "numeric" and is_numeric_dtype(df[col]):
          mismatch = False

      if mismatch:

          print("\n--------------------------------------")
          print(f"Column          : {col}")
          print(f"Current Type    : {current_dtype}")
          print(f"Numeric Match   : {round(numeric_ratio*100,2)}%")
          print(f"Datetime Match  : {round(datetime_ratio*100,2)}%")
          print(f"Suggested Type  : {suggested_type}")
          print("--------------------------------------")

          if input("Convert to suggested type? (y/n): ").lower() == "y":

              try:
                  if suggested_type == "numeric":
                      df[col] = pd.to_numeric(cleaned, errors="coerce")
                  elif suggested_type == "datetime":
                      df[col] = pd.to_datetime(df[col], errors="coerce")
                  elif suggested_type == "time":
                      df[col] = pd.to_datetime(df[col], errors="coerce")
                  elif suggested_type == "string":
                      df[col] = df[col].astype(str)

                  log_action(
                      f"{col} converted from {current_dtype} "
                      f"to {suggested_type} (data-driven decision)."
                  )
                  report["datatype_changes"].append({
                      "column": col,
                      "current_type": current_dtype,
                      "suggested": suggested_type,
                      "action": "converted",
                      "reason": f"{round(max(numeric_ratio, datetime_ratio)*100,1)}% value compatibility"
                  })
                  print("Conversion applied.")

              except Exception as e:
                  print("Conversion failed:", e)
                  report["datatype_changes"].append({
                      "column": col,
                      "action": f"conversion FAILED — {e}"
                  })
          else:
              log_action(f"{col} conversion skipped by user.")
              report["datatype_changes"].append({
                  "column": col,
                  "current_type": current_dtype,
                  "suggested": suggested_type,
                  "action": "skipped — user declined",
                  "reason": "user chose not to convert"
              })
              print("Conversion skipped.")

      else:

          print(f"{col:<35} {current_dtype:<15} → {suggested_type:<15} already correct")
          report["datatype_changes"].append({
              "column" : col,
              "current_type": current_dtype,
              "suggested": suggested_type,
              "action": "no change needed",
              "reason": "current type matches detected type"
          })

  integrity_snapshot("Datatype Review", df, original_row_count)

  print("\n=========== MISSING VALUE HANDLING ===========")

  print("""
Missing value strategy:
• Numeric columns → Mean or Median
• Text columns → Fill with 'NULL'
• ID columns → Fill with 'NULL' only (no statistical filling)
• Datetime columns → Optional forward fill or skip
""")

  for col in df.columns:

      missing_count = df[col].isnull().sum()

      if missing_count == 0:
          continue

      print("\n--------------------------------------")
      print(f"Column: {col}")
      print(f"Missing Values: {missing_count}")
      print(f"Data Type: {df[col].dtype}")
      print("--------------------------------------")

      if col in id_columns:
          print("ID column detected.")
          if input("Fill missing ID values with 'NULL'? (y/n): ").lower() == "y":
              df[col] = df[col].fillna("NULL")
              log_action(f"{col} (ID) filled with 'NULL'.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "filled with NULL", "type": "ID"
              })
          else:
              log_action(f"{col} (ID) missing values left unchanged.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "left unchanged", "type": "ID"
              })
          continue

      if is_numeric_dtype(df[col]):
          print("1 - Fill with Median (recommended for skewed data)")
          print("2 - Fill with Mean (recommended for normal data)")
          print("3 - Skip")

          choice = input("Choose: ")

          if choice == "1":
              val = df[col].median()
              df[col] = df[col].fillna(val)
              log_action(f"{col} filled with median.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": f"median ({round(val,4)})", "type": "numeric"
              })
          elif choice == "2":
              val = df[col].mean()
              df[col] = df[col].fillna(val)
              log_action(f"{col} filled with mean.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": f"mean ({round(val,4)})", "type": "numeric"
              })
          else:
              log_action(f"{col} numeric missing values skipped.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "skipped by user", "type": "numeric"
              })

      elif is_datetime64_any_dtype(df[col]):
          print("1 - Forward Fill")
          print("2 - Fill with NaT")
          print("3 - Skip")

          choice = input("Choose: ")

          if choice == "1":
              df[col] = df[col].ffill()
              log_action(f"{col} forward-filled.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "forward fill", "type": "datetime"
              })
          elif choice == "2":
              df[col] = df[col].fillna(pd.NaT)
              log_action(f"{col} datetime missing filled with NaT.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "filled with NaT", "type": "datetime"
              })
          else:
              log_action(f"{col} datetime missing values skipped.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "skipped by user", "type": "datetime"
              })

      else:
          print("Text column detected.")
          if input("Fill missing values with 'NULL'? (y/n): ").lower() == "y":
              df[col] = df[col].fillna("NULL")
              log_action(f"{col} text missing values filled with 'NULL'.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "filled with NULL", "type": "text"
              })
          else:
              log_action(f"{col} text missing values skipped.")
              report["missing_handling"].append({
                  "column": col, "missing_count": int(missing_count),
                  "strategy": "skipped by user", "type": "text"
              })

  integrity_snapshot("Missing Value Handling", df, original_row_count)

  trimmed_cols   = []
  trimmed_counts = {}

  for col in df.columns:
      if str(df[col].dtype) not in ["object", "string", "category"]:
          continue
      if col in id_columns:
          continue
      before         = df[col].copy()
      df[col]        = df[col].str.strip() if hasattr(df[col], "str") else df[col]
      changed        = (df[col].fillna("") != before.fillna("")).sum()
      if changed > 0:
          trimmed_cols.append(col)
          trimmed_counts[col] = int(changed)
          log_action(f"{col} — {changed} cell(s) had leading/trailing whitespace trimmed.")

  if trimmed_cols:
      print(f"\n✔ Whitespace trimmed in {len(trimmed_cols)} column(s):")
      for col in trimmed_cols:
          print(f"   {col}: {trimmed_counts[col]} cell(s) trimmed")
  else:
      print("\n✔ Whitespace check complete — no trimming needed.")

  report["whitespace_trim"] = {
      "columns_trimmed": trimmed_cols,
      "counts"         : trimmed_counts,
  }

  integrity_snapshot("Whitespace Trimming", df, original_row_count)

  print("\n=========== TEXT NORMALIZATION ===========")

  print("""
This step allows controlled normalization of text columns.

Available transformations:
1 - Remove underscores (_)
2 - Remove dashes (-)
3 - Lowercase
4 - Title Case
5 - Uppercase

You may enter multiple options separated by comma.
Example: 1,3
""")

  for col in df.columns:

      if col in id_columns:
          continue

      if not is_string_dtype(df[col]):
          continue

      series = df[col].dropna()

      if len(series) == 0:
          continue

      print("\n--------------------------------------")
      print(f"Column: {col}")
      print(f"Unique Values: {series.nunique()}")

      top_vals = series.astype(str).value_counts().head(10)
      print("Top 10 Values:")
      print(top_vals.to_dict())
      print("--------------------------------------")

      contains_underscore = series.astype(str).str.contains("_").any()
      contains_dash       = series.astype(str).str.contains("-").any()
      mixed_case          = not (
          series.astype(str).str.islower().all() or
          series.astype(str).str.isupper().all()
      )

      print("Detected Patterns:")
      if contains_underscore: print(" • Contains underscores (_)")
      if contains_dash:       print(" • Contains dashes (-)")
      if mixed_case:          print(" • Mixed casing detected")

      print("\nOptions:")
      print("1 - Remove underscores (_)")
      print("2 - Remove dashes (-)")
      print("3 - Lowercase")
      print("4 - Title Case")
      print("5 - Uppercase")
      print("6 - Skip")

      choice = input("Enter options (comma separated): ").strip()

      ops_applied = []
      if choice:
          operations = [x.strip() for x in choice.split(",")]

          for op in operations:
              if op == "1":
                  df[col] = df[col].str.replace("_", " ", regex=False)
                  ops_applied.append("remove underscores")
              elif op == "2":
                  df[col] = df[col].str.replace("-", " ", regex=False)
                  ops_applied.append("remove dashes")
              elif op == "3":
                  df[col] = df[col].str.lower()
                  ops_applied.append("lowercase")
              elif op == "4":
                  df[col] = df[col].str.title()
                  ops_applied.append("title case")
              elif op == "5":
                  df[col] = df[col].str.upper()
                  ops_applied.append("uppercase")
              elif op == "6":
                  ops_applied.append("skipped")

          if ops_applied and ops_applied != ["skipped"]:
              log_action(f"{col} text normalized using operations: {choice}")
              report["text_normalization"].append({
                  "column"     : col,
                  "operations" : ops_applied,
                  "action"     : "applied"
              })
              print("Transformation applied.")
          else:
              report["text_normalization"].append({
                  "column": col, "operations": [], "action": "skipped by user"
              })

  integrity_snapshot("Text Normalization", df, original_row_count)

  print("\n=========== FINAL VALIDATION ===========")

  final_row_count     = df.shape[0]
  final_columns_list  = list(df.columns)
  final_column_set    = set(df.columns)
  final_dtypes        = df.dtypes.astype(str).to_dict()

  report["final"]["rows"]    = final_row_count
  report["final"]["cols"]    = len(df.columns)
  report["final"]["missing"] = int(df.isnull().sum().sum())
  report["final"]["dupes"]   = int(df.duplicated().sum())

  print("\n--- ROW CHECK ---")
  print(f"Original rows: {original_row_count}")
  print(f"Final rows: {final_row_count}")
  if final_row_count != original_row_count:
      print(f"Row difference: {original_row_count - final_row_count}")

  print("\n--- COLUMN CHECK ---")
  added_columns   = final_column_set - original_column_set
  removed_columns = original_column_set - final_column_set

  if added_columns:
      print("Added columns detected:", added_columns)
  if removed_columns:
      print("Removed columns detected:", removed_columns)
  if not added_columns and not removed_columns:
      print("No columns added or removed.")

  print("\n--- COLUMN ORDER CHECK ---")
  if original_columns_list != final_columns_list:
      print("Column order has changed.")
  else:
      print("Column order unchanged.")


  print("\n--- DATATYPE CHANGES ---")
  datatype_changes = []
  for col in original_dtypes:
      if col in final_dtypes:
          if original_dtypes[col] != final_dtypes[col]:
              datatype_changes.append((col, original_dtypes[col], final_dtypes[col]))

  if datatype_changes:
      for col, old_type, new_type in datatype_changes:
          print(f"{col}: {old_type} → {new_type}")
  else:
      print("No datatype changes detected.")

  unexpected_additions = added_columns - {"is_duplicate"}
  if unexpected_additions:
      raise ValueError("Unexpected column addition detected!")
  if removed_columns:
      raise ValueError("Unexpected column removal detected!")
  if final_row_count != original_row_count and not rows_removed:
      raise ValueError("Unexpected row change detected!")

  print("\n✔ Final integrity validation passed.")


  base_name, extension = os.path.splitext(file_name)
  audit_file_name = f"{base_name}_audit_report.txt"

  lines = []
  lines.append("=" * 65)
  lines.append("DATA CLEANING AUDIT REPORT")
  lines.append(f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
  lines.append("=" * 65)

  lines.append("\n── SECTION 1: FILE INFORMATION ──")
  lines.append(f"File name: {report['file_info'].get('filename', 'N/A')}")
  lines.append(f"Loaded at: {report['file_info'].get('loaded_at', 'N/A')}")
  if "encoding" in report["file_info"]:
      lines.append(f"Encoding: {report['file_info']['encoding']}")

  lines.append("\n── SECTION 2: SHAPE SUMMARY ──")
  lines.append(f"Original rows    : {report['shape']['original_rows']:,}")
  lines.append(f"Original columns : {report['shape']['original_cols']}")
  lines.append(f"Final rows       : {report['final']['rows']:,}")
  lines.append(f"Final columns    : {report['final']['cols']}")
  diff = report['shape']['original_rows'] - report['final']['rows']
  if diff > 0:
      lines.append(f"Rows removed     : {diff:,} (user approved)")
  else:
      lines.append(f"Rows removed     : 0 — no rows were removed")
  lines.append(f"Final missing    : {report['final']['missing']:,} values")
  lines.append(f"Final duplicates : {report['final']['dupes']:,} rows")


  lines.append("\n── SECTION 3: COLUMN STANDARDIZATION ──")
  if report["column_std"].get("applied"):
      renamed = report["column_std"].get("renamed", [])
      lines.append(f"Applied: Yes — {len(renamed)} column(s) renamed")
      lines.append(f"Format: lowercase snake_case")
      if renamed:
          lines.append("  Changes  :")
          for old, new in renamed:
              lines.append(f"    '{old}'  →  '{new}'")
  else:
      lines.append("  Applied  : No — skipped by user")


  lines.append("\n── SECTION 4: ID COLUMN PROTECTION ──")
  if report["id_protection"]:
      for entry in report["id_protection"]:
          status = "PROTECTED" if entry["protected"] else "NOT protected"
          reason = entry.get("reason", "")
          lines.append(f"  {entry['column']:<35} {status} | {reason}")
  else:
      lines.append("  No ID column candidates detected.")


  lines.append("\n── SECTION 5: DUPLICATE HANDLING ──")
  d = report["duplicates"]
  lines.append(f"  Duplicates found : {d.get('count_found', 0):,}")
  lines.append(f"  Action taken     : {d.get('action', 'N/A')}")
  if d.get("rows_removed", 0) > 0:
      lines.append(f"  Rows removed     : {d['rows_removed']:,}")
      lines.append(f"  Rows before      : {d['rows_before']:,}")
      lines.append(f"  Rows after       : {d['rows_after']:,}")
  if d.get("flag_column"):
      lines.append(f"  Flag column      : {d['flag_column']}")


  lines.append("\n── SECTION 6: DATATYPE CHANGES ──")
  lines.append(f"  {'Column':<35} {'Before':<15} {'Suggested':<15} {'Action'}")
  lines.append(f"  {'-'*80}")

  converted_cols = []
  skipped_cols   = []
  no_change_cols = []

  for entry in report["datatype_changes"]:
      col     = entry.get("column", "")
      before  = entry.get("current_type", "—")
      suggest = entry.get("suggested", "—")
      action  = entry.get("action", "—")
      reason  = entry.get("reason", "")

      lines.append(f"  {col:<35} {before:<15} {suggest:<15} {action}")
      if reason:
          lines.append(f"    └─ Reason: {reason}")


      if action == "converted":
          converted_cols.append(col)
      elif "skipped" in action or "declined" in action:
          skipped_cols.append(col)
      elif "no change" in action or "protected" in action:
          no_change_cols.append(col)

  lines.append(f"\n  Summary:")
  lines.append(f"Columns converted          : {len(converted_cols)}")
  if converted_cols:
      for c in converted_cols:
          lines.append(f"      - {c}")
  lines.append(f"Columns skipped by user: {len(skipped_cols)}")
  if skipped_cols:
      for c in skipped_cols:
          lines.append(f" - {c}")
  lines.append(f"Columns already correct: {len(no_change_cols)}")
  if no_change_cols:
      for c in no_change_cols:
          lines.append(f"- {c}")


  lines.append("\n── SECTION 7: MISSING VALUE HANDLING ──")
  if report["missing_handling"]:
      lines.append(f"  {'Column':<35} {'Type':<12} {'Missing':>10}  {'Strategy'}")
      lines.append(f"  {'-'*75}")
      for entry in report["missing_handling"]:
          lines.append(
              f"  {entry['column']:<35} {entry['type']:<12} "
              f"{entry['missing_count']:>10,}  {entry['strategy']}"
          )
  else:
      lines.append("  No missing values found in dataset.")


  lines.append("\n── SECTION 7b: CELL-LEVEL WHITESPACE TRIMMING ──")
  wt = report.get("whitespace_trim", {})
  trimmed = wt.get("columns_trimmed", [])
  counts  = wt.get("counts", {})
  if trimmed:
      lines.append(f"  Columns affected : {len(trimmed)}")
      for col in trimmed:
          lines.append(f"    {col:<35} {counts.get(col, 0):,} cell(s) trimmed")
  else:
      lines.append("  No whitespace trimming needed — all cells already clean.")


  lines.append("\n── SECTION 8: TEXT NORMALIZATION ──")
  if report["text_normalization"]:
      for entry in report["text_normalization"]:
          ops = ", ".join(entry["operations"]) if entry["operations"] else "none"
          lines.append(f"  {entry['column']:<35} {entry['action']} | ops: {ops}")
  else:
      lines.append("  No text normalization applied.")

  lines.append("\n── SECTION 9: FINAL VALIDATION ──")
  if datatype_changes:
      lines.append("  Confirmed datatype changes:")
      for col, old, new in datatype_changes:
          lines.append(f"    {col}: {old} → {new}")
  else:
      lines.append("  No datatype changes confirmed.")
  lines.append("  Integrity check : PASSED")
  lines.append("  Unexpected column additions : " +
               ("None" if not unexpected_additions else str(unexpected_additions)))

  lines.append("\n── SECTION 10: FULL ACTION LOG ──")
  for entry in audit_log:
      lines.append(f"  {entry}")

  lines.append("\n" + "=" * 65)
  lines.append("  CLEANING COMPLETE")
  lines.append("=" * 65)

  full_report = "\n".join(lines)
  print("\n" + full_report)

  with open(audit_file_name, "w") as f:
      f.write(full_report)

  from openpyxl.utils import get_column_letter
  import openpyxl as _openpyxl
  from pandas.api.types import is_integer_dtype, is_float_dtype

  excel_file_name = f"{base_name}_cleaned.xlsx"

  with pd.ExcelWriter(excel_file_name, engine='openpyxl') as writer:
      df.to_excel(writer, index=False)

  wb = _openpyxl.load_workbook(excel_file_name)
  ws = wb.active

  format_applied = {}

  for i, (col, dtype) in enumerate(df.dtypes.items()):
      col_idx    = i + 1
      col_letter = get_column_letter(col_idx)
      dtype_str  = str(dtype)

      series = df[col].dropna()

      if 'datetime' in dtype_str:
          fmt = 'dd-mm-yyyy'

      elif is_numeric_dtype(df[col]) and is_integer_dtype(df[col]):
          fmt = '0'

      elif is_numeric_dtype(df[col]) and is_float_dtype(df[col]):


          if len(series) > 0 and series.between(0, 1).mean() > 0.8:
              fmt = '0.00%'
          else:
              fmt = '#,##0.00'


      elif dtype_str in ['object', 'string', 'category']:
          fmt = '@'

      else:
          fmt = 'General'


      for row in range(2, len(df) + 2):
          ws[f'{col_letter}{row}'].number_format = fmt

      format_applied[col] = fmt

  wb.save(excel_file_name)

  print("\n  Excel column formats applied:")
  print(f"  {'Column':<35} {'Pandas Type':<15} {'Excel Format'}")
  print(f"  {'-'*65}")
  for i, (col, dtype) in enumerate(df.dtypes.items()):
      print(f"  {col:<35} {str(dtype):<15} {format_applied[col]}")

  print(f"\nCleaned Dataset Saved : {excel_file_name}")
  print(f"Audit Report Saved    : {audit_file_name}")
  print("\n========== CLEANING COMPLETE ==========")

  try:
      from google.colab import files
      files.download(excel_file_name)
      files.download(audit_file_name)
  except Exception:
      pass


run_data_cleaner()


Upload your dataset (CSV, Excel, JSON, Parquet, TXT supported)...
